In [1]:
from src.data import load_raw, clean, split_by_date
from src.features import prepare_features, TARGET
from sklearn.metrics import root_mean_squared_error
from sklearn.ensemble import RandomForestRegressor
import numpy as np

df = clean(load_raw())
train, val = split_by_date(df)
train = train.sample(500_000, random_state=42)

X_train, y_train = prepare_features(train), train[TARGET]
X_val, y_val = prepare_features(val), val[TARGET]

# Baseline: always predict the mean
baseline_rmse = root_mean_squared_error(y_val, np.full(len(y_val), y_train.mean()))

# Random forest
rf = RandomForestRegressor(n_estimators=100, max_depth=20, min_samples_leaf=5,
                           n_jobs=-1, random_state=42).fit(X_train, y_train)
rf_rmse = root_mean_squared_error(y_val, rf.predict(X_val))

print(f"Baseline RMSE: {baseline_rmse:.2f} | RF RMSE: {rf_rmse:.2f}")

Baseline RMSE: 9.56 | RF RMSE: 4.09


In [2]:
import mlflow
import time
import joblib
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score

from src.config import CFG, PROJECT_ROOT
from src.data import load_raw, clean, split_by_date
from src.features import prepare_features, FEATURES, TARGET

ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
ARTIFACTS_DIR.mkdir(exist_ok=True)

mlflow.set_tracking_uri(CFG["mlflow"]["tracking_uri"])
mlflow.set_experiment(CFG["mlflow"]["experiment"])     # created if it doesn't exist

# Data
df = clean(load_raw())
train, val = split_by_date(df)
train = train.sample(CFG["split"]["train_sample"], random_state=42)
X_train, y_train = prepare_features(train), train[TARGET]
X_val, y_val = prepare_features(val), val[TARGET]

params = {**CFG["model"], "n_jobs": -1}

with mlflow.start_run(run_name="rf-baseline max_depth=10") as run:
    # Tags:
    mlflow.set_tags({
        "model_type": "random_forest",
        "stage": "baseline - depth 10",
        "data_period": f"{CFG['run']['year']}-{CFG['run']['months'][0]:02d}",
    })
    # Parameters: 
    mlflow.log_params(params)
    mlflow.log_params({
        "train_rows": len(X_train),
        "val_rows": len(X_val),
        "cutoff": str(CFG["split"]["cutoff"]),
        "features": ",".join(FEATURES),
    })

    # Train
    t0 = time.perf_counter()
    rf = RandomForestRegressor(**params).fit(X_train, y_train)
    train_seconds = time.perf_counter() - t0

    t0 = time.perf_counter()
    y_pred = rf.predict(X_val)
    predict_ms_per_row = (time.perf_counter() - t0) * 1000 / len(X_val)
    

    #  Metrics: how well it did
    baseline_rmse = root_mean_squared_error(y_val, np.full(len(y_val), y_train.mean()))
    mlflow.log_metrics({
        "val_rmse": root_mean_squared_error(y_val, y_pred),
        "val_mae": mean_absolute_error(y_val, y_pred),
        "val_r2": r2_score(y_val, y_pred),
        "train_rmse": root_mean_squared_error(y_train, rf.predict(X_train)),
        "baseline_rmse": baseline_rmse,
    })

    model_path = ARTIFACTS_DIR / "model.joblib"
    joblib.dump(rf, model_path)
    model_size_mb = model_path.stat().st_size / 1e6

    # Cost metrics                                              # NEW
    mlflow.log_metrics({
        "train_seconds": train_seconds,
        "predict_ms_per_row": predict_ms_per_row,
        "model_size_mb": model_size_mb,
    })

    # The model itself
    mlflow.sklearn.log_model(
        rf, name="model",
        input_example=X_val.head(3),
        skops_trusted_types=["sklearn.tree._tree.Tree"],
    )

d:\Malar\Learning\AI_MachineLearning\Materials\Project\mle-model-deployment-project\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/11/22 17:50:15 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


🏃 View run rf-baseline max_depth=10 at: http://127.0.0.1:5000/#/experiments/1/runs/88320115930d4716bf69be4dffd21160
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
